# Notebook 1. Setup and Data

This Jupyter notebook provides the dataset on which our analysis will be based.

1. Create seed datasets for all 50 states and DC
2. Initialize the SQLite database according to db/schema.sql
3. Fill out the dimension tables (state and age_groups)
4. Import CSV files into the six data tables

It should be run only once, before the next notebook (02_projection_pipeline.ipynb).

The synthetic values come from actual US demographic averages. In cases where we have actual figures, such as from Census, CDC WONDER, BLS, SEER, and ACS, we just have to replace the cleaned CSVs with those that share the same filenames in Section 2.

In [1]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DB_PATH = PROJECT_ROOT / 'db' / 'projection.db'
SCHEMA_PATH = PROJECT_ROOT / 'db' / 'schema.sql'

RNG = np.random.default_rng(seed=210)

## 1. Synthetic rate and population data

All values are approximations of US national averages with small differences from one state to another. Every state produces a noticeably different projection instead of the same curve. We keep all the constants at the top of the cell so they are easy to find and defend.

### 1.1 Reference constants

In [2]:
STATES = [
    ('AL','Alabama','South','East South Central'),('AK','Alaska','West','Pacific'),
    ('AZ','Arizona','West','Mountain'),('AR','Arkansas','South','West South Central'),
    ('CA','California','West','Pacific'),('CO','Colorado','West','Mountain'),
    ('CT','Connecticut','Northeast','New England'),('DE','Delaware','South','South Atlantic'),
    ('DC','District of Columbia','South','South Atlantic'),('FL','Florida','South','South Atlantic'),
    ('GA','Georgia','South','South Atlantic'),('HI','Hawaii','West','Pacific'),
    ('ID','Idaho','West','Mountain'),('IL','Illinois','Midwest','East North Central'),
    ('IN','Indiana','Midwest','East North Central'),('IA','Iowa','Midwest','West North Central'),
    ('KS','Kansas','Midwest','West North Central'),('KY','Kentucky','South','East South Central'),
    ('LA','Louisiana','South','West South Central'),('ME','Maine','Northeast','New England'),
    ('MD','Maryland','South','South Atlantic'),('MA','Massachusetts','Northeast','New England'),
    ('MI','Michigan','Midwest','East North Central'),('MN','Minnesota','Midwest','West North Central'),
    ('MS','Mississippi','South','East South Central'),('MO','Missouri','Midwest','West North Central'),
    ('MT','Montana','West','Mountain'),('NE','Nebraska','Midwest','West North Central'),
    ('NV','Nevada','West','Mountain'),('NH','New Hampshire','Northeast','New England'),
    ('NJ','New Jersey','Northeast','Mid-Atlantic'),('NM','New Mexico','West','Mountain'),
    ('NY','New York','Northeast','Mid-Atlantic'),('NC','North Carolina','South','South Atlantic'),
    ('ND','North Dakota','Midwest','West North Central'),('OH','Ohio','Midwest','East North Central'),
    ('OK','Oklahoma','South','West South Central'),('OR','Oregon','West','Pacific'),
    ('PA','Pennsylvania','Northeast','Mid-Atlantic'),('RI','Rhode Island','Northeast','New England'),
    ('SC','South Carolina','South','South Atlantic'),('SD','South Dakota','Midwest','West North Central'),
    ('TN','Tennessee','South','East South Central'),('TX','Texas','South','West South Central'),
    ('UT','Utah','West','Mountain'),('VT','Vermont','Northeast','New England'),
    ('VA','Virginia','South','South Atlantic'),('WA','Washington','West','Pacific'),
    ('WV','West Virginia','South','South Atlantic'),('WI','Wisconsin','Midwest','East North Central'),
    ('WY','Wyoming','West','Mountain'),
]

AGE_GROUPS = [
    ('0-4',0,4,1),('5-9',5,9,2),('10-14',10,14,3),('15-19',15,19,4),
    ('20-24',20,24,5),('25-29',25,29,6),('30-34',30,34,7),('35-39',35,39,8),
    ('40-44',40,44,9),('45-49',45,49,10),('50-54',50,54,11),('55-59',55,59,12),
    ('60-64',60,64,13),('65-69',65,69,14),('70-74',70,74,15),('75-79',75,79,16),
    ('80-84',80,84,17),('85+',85,None,18),
]

STATE_POP_MILLIONS = {
    'AL':5.1,'AK':0.73,'AZ':7.4,'AR':3.1,'CA':39.0,'CO':5.9,'CT':3.6,'DE':1.0,
    'DC':0.68,'FL':22.6,'GA':11.0,'HI':1.4,'ID':2.0,'IL':12.5,'IN':6.8,'IA':3.2,
    'KS':2.9,'KY':4.5,'LA':4.6,'ME':1.4,'MD':6.2,'MA':7.0,'MI':10.0,'MN':5.7,
    'MS':2.9,'MO':6.2,'MT':1.1,'NE':2.0,'NV':3.2,'NH':1.4,'NJ':9.3,'NM':2.1,
    'NY':19.6,'NC':10.8,'ND':0.78,'OH':11.8,'OK':4.1,'OR':4.2,'PA':13.0,'RI':1.1,
    'SC':5.3,'SD':0.92,'TN':7.1,'TX':30.5,'UT':3.4,'VT':0.65,'VA':8.7,'WA':7.8,
    'WV':1.8,'WI':5.9,'WY':0.58,
}

RACE_ORIGIN_SHARES = {
    ('white','non-hisp'):0.57, ('white','hisp'):0.12,
    ('black','non-hisp'):0.12, ('black','hisp'):0.01,
    ('asian','non-hisp'):0.06, ('asian','hisp'):0.002,
    ('other','non-hisp'):0.10, ('other','hisp'):0.018,
}

AGE_SHARES = {
    '0-4':0.058,'5-9':0.060,'10-14':0.064,'15-19':0.064,'20-24':0.066,
    '25-29':0.070,'30-34':0.070,'35-39':0.068,'40-44':0.063,'45-49':0.060,
    '50-54':0.064,'55-59':0.067,'60-64':0.064,'65-69':0.054,'70-74':0.045,
    '75-79':0.032,'80-84':0.020,'85+':0.021,
}

ANNUAL_FERTILITY_BY_AGE = {
    '15-19':0.015,'20-24':0.065,'25-29':0.100,'30-34':0.100,
    '35-39':0.050,'40-44':0.012,'45-49':0.001,
}

MORTALITY_5YR_BY_AGE = {
    '0-4':0.0025,'5-9':0.0005,'10-14':0.0005,'15-19':0.0025,'20-24':0.0040,
    '25-29':0.0050,'30-34':0.0065,'35-39':0.0090,'40-44':0.0124,'45-49':0.0184,
    '50-54':0.0272,'55-59':0.0394,'60-64':0.0586,'65-69':0.0868,'70-74':0.1324,
    '75-79':0.2058,'80-84':0.3226,'85+':0.5563,
}

LFPR_BY_AGE_SEX = {
    ('0-4','m'):0.0,('0-4','f'):0.0,('5-9','m'):0.0,('5-9','f'):0.0,
    ('10-14','m'):0.0,('10-14','f'):0.0,
    ('15-19','m'):0.36,('15-19','f'):0.32,('20-24','m'):0.75,('20-24','f'):0.70,
    ('25-29','m'):0.88,('25-29','f'):0.75,('30-34','m'):0.91,('30-34','f'):0.73,
    ('35-39','m'):0.91,('35-39','f'):0.74,('40-44','m'):0.90,('40-44','f'):0.77,
    ('45-49','m'):0.87,('45-49','f'):0.77,('50-54','m'):0.83,('50-54','f'):0.74,
    ('55-59','m'):0.75,('55-59','f'):0.65,('60-64','m'):0.60,('60-64','f'):0.50,
    ('65-69','m'):0.38,('65-69','f'):0.28,('70-74','m'):0.23,('70-74','f'):0.15,
    ('75-79','m'):0.12,('75-79','f'):0.07,('80-84','m'):0.06,('80-84','f'):0.03,
    ('85+','m'):0.02,('85+','f'):0.01,
}

EMP_TO_POP_RATIO = 0.48
ANNUAL_EMP_GROWTH = {
    'TX':0.014,'FL':0.013,'AZ':0.012,'NV':0.012,'NC':0.011,'GA':0.010,
    'SC':0.010,'TN':0.010,'UT':0.011,'CO':0.010,'ID':0.012,'WA':0.010,
    'CA':0.006,'NY':0.004,'NJ':0.005,'MA':0.006,'MD':0.006,'VA':0.007,
    'OR':0.008,'MN':0.006,'WI':0.005,'PA':0.003,'WV':-0.002,'IL':0.002,
    'MS':0.003,'LA':0.002,'OH':0.003,'MI':0.003,'AR':0.004,'OK':0.005,
    'AL':0.004,'KY':0.004,'MO':0.004,'IA':0.004,'KS':0.004,'NE':0.005,
    'IN':0.004,'RI':0.003,'CT':0.003,'NH':0.005,'VT':0.002,'ME':0.003,
    'MT':0.006,'WY':0.003,'NM':0.004,'AK':0.002,'HI':0.005,'ND':0.004,
    'SD':0.005,'DE':0.005,'DC':0.004,
}

### 1.2 Builders, one per rate table

Each builder returns a DataFrame in the exact shape of the matching SQL table.

In [3]:
def build_population(year):
    rows = []
    for state_code, *_ in STATES:
        total = STATE_POP_MILLIONS[state_code] * 1_000_000
        if year == 2018:
            total *= 0.97
        for (race, origin), ro_share in RACE_ORIGIN_SHARES.items():
            for age_group, ag_share in AGE_SHARES.items():
                for sex in ('m', 'f'):
                    sex_share = 0.50
                    if age_group in ('80-84', '85+'):
                        sex_share = 0.40 if sex == 'm' else 0.60
                    cell = total * ro_share * ag_share * sex_share
                    cell *= 1 + RNG.normal(0, 0.03)
                    rows.append({
                        'state_code': state_code, 'year': year,
                        'race': race, 'origin': origin, 'sex': sex,
                        'age_group': age_group,
                        'population': max(int(round(cell)), 0),
                        'is_projection': 0,
                    })
    return pd.DataFrame(rows)


def build_fertility_rates(year=2023):
    rows = []
    for state_code, _, region, _ in STATES:
        region_adj = {'Northeast':0.92,'Midwest':0.98,'South':1.05,'West':1.00}[region]
        for (race, origin), _ in RACE_ORIGIN_SHARES.items():
            race_adj = 1.15 if origin == 'hisp' else (1.08 if race == 'other' else 1.0)
            for age_group in AGE_SHARES.keys():
                base = ANNUAL_FERTILITY_BY_AGE.get(age_group, 0.0)
                rate = base * region_adj * race_adj
                rows.append({
                    'state_code':state_code,'year':year,
                    'race':race,'origin':origin,'age_group':age_group,
                    'fertility_rate':round(rate, 6),'source':'synthetic_seed',
                })
    return pd.DataFrame(rows)


def build_mortality_rates(year=2023):
    rows = []
    for state_code, _, region, _ in STATES:
        region_adj = {'Northeast':0.95,'Midwest':1.02,'South':1.08,'West':0.97}[region]
        for (race, origin), _ in RACE_ORIGIN_SHARES.items():
            race_adj = 1.10 if race == 'black' else (0.95 if race == 'asian' else 1.0)
            for sex in ('m','f'):
                sex_adj = 1.20 if sex == 'm' else 0.85
                for age_group in AGE_SHARES.keys():
                    base = MORTALITY_5YR_BY_AGE[age_group]
                    rate = min(base * region_adj * race_adj * sex_adj, 0.999)
                    rows.append({
                        'state_code':state_code,'year':year,
                        'race':race,'origin':origin,'sex':sex,'age_group':age_group,
                        'mortality_rate':round(rate, 6),'source':'synthetic_seed',
                    })
    return pd.DataFrame(rows)


def build_lfpr(year=2023):
    rows = []
    for state_code, *_ in STATES:
        for (race, origin), _ in RACE_ORIGIN_SHARES.items():
            race_adj = 1.02 if race == 'asian' else (0.97 if race == 'black' else 1.0)
            for age_group in AGE_SHARES.keys():
                for sex in ('m','f'):
                    base = LFPR_BY_AGE_SEX[(age_group, sex)]
                    rate = max(0.0, min(1.0, base * race_adj * (1 + RNG.normal(0, 0.02))))
                    rows.append({
                        'state_code':state_code,'year':year,
                        'race':race,'origin':origin,'sex':sex,'age_group':age_group,
                        'lfpr':round(rate, 4),'source':'synthetic_seed',
                    })
    return pd.DataFrame(rows)


def build_employment_projections():
    rows = []
    years = [2023, 2028, 2033, 2038, 2043, 2048, 2053, 2058]
    for state_code, *_ in STATES:
        base_2023 = STATE_POP_MILLIONS[state_code] * 1_000_000 * EMP_TO_POP_RATIO
        g = ANNUAL_EMP_GROWTH.get(state_code, 0.005)
        for y in years:
            emp = base_2023 * (1 + g) ** (y - 2023)
            rows.append({
                'state_code':state_code,'year':y,
                'total_employment':max(int(round(emp)), 0),
                'source':'synthetic_seed',
            })
    return pd.DataFrame(rows)


def build_migration_estimates(year=2023):
    rows = []
    for state_code, *_ in STATES:
        growth = ANNUAL_EMP_GROWTH.get(state_code, 0.005)
        state_signal = np.clip(growth * 2.0, -0.02, 0.03)
        for (race, origin), _ in RACE_ORIGIN_SHARES.items():
            for sex in ('m','f'):
                for age_group in AGE_SHARES.keys():
                    age_weight = (
                        1.0 if age_group in ('20-24','25-29','30-34','35-39','40-44')
                        else 0.5 if age_group in ('15-19','45-49','50-54')
                        else 0.2
                    )
                    rate = state_signal * age_weight * (1 + RNG.normal(0, 0.15))
                    rows.append({
                        'state_code':state_code,'year':year,
                        'race':race,'origin':origin,'sex':sex,'age_group':age_group,
                        'net_migration':None,'migration_rate':round(float(rate), 5),
                        'source':'synthetic_seed',
                    })
    return pd.DataFrame(rows)

### 1.3 Write CSVs to data

In [4]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.ingest.census import build_population
from src.ingest.bls import build_employment_projections

for yr in (2018, 2023):
    df_pop = build_population(yr)
    df_pop.to_csv(DATA_DIR / f"population_{yr}.csv", index=False)
    print(f"Replaced population_{yr}.csv with Census ACS data: {len(df_pop):,} rows, {df_pop.population.sum():,} total")

emp_real = build_employment_projections()
emp_real.to_csv(DATA_DIR / "employment_projections.csv", index=False)
print(f"Replaced employment_projections.csv with BLS data: {len(emp_real):,} rows")

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("DELETE FROM population WHERE is_projection=0")
    conn.execute("DELETE FROM population WHERE is_projection=1")
    for yr in (2018, 2023):
        df_load = pd.read_csv(DATA_DIR / f"population_{yr}.csv")
        conn.executemany(
            "INSERT INTO population (state_code,year,race,origin,sex,age_group,population,is_projection) VALUES (?,?,?,?,?,?,?,?)",
            list(df_load.itertuples(index=False, name=None)),
        )
    conn.execute("DELETE FROM employment_projections")
    conn.executemany(
        "INSERT INTO employment_projections (state_code,year,total_employment,source) VALUES (?,?,?,?)",
        list(emp_real.itertuples(index=False, name=None)),
    )
    conn.commit()
print("Real data loaded into DB.")

Replaced population_2018.csv with Census ACS data: 14,688 rows, 330,439,019 total
Replaced population_2023.csv with Census ACS data: 14,688 rows, 338,263,983 total
Replaced employment_projections.csv with BLS data: 408 rows
Real data loaded into DB.


## 2. Initialize the SQLite database

We read the DDL from db/schema.sql and apply it. Then we populate the two dimension tables. This is safe to rerun because it uses CREATE TABLE IF NOT EXISTS and INSERT OR IGNORE.

In [5]:
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

with sqlite3.connect(DB_PATH) as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    conn.executescript(SCHEMA_PATH.read_text(encoding='utf-8'))
    conn.executemany(
        'INSERT OR IGNORE INTO states (state_code, state_name, region, division) VALUES (?, ?, ?, ?)',
        STATES,
    )
    conn.executemany(
        'INSERT OR IGNORE INTO age_groups (age_group, age_min, age_max, sort_order) VALUES (?, ?, ?, ?)',
        AGE_GROUPS,
    )
    conn.commit()

print(f'Initialized {DB_PATH}  |  {len(STATES)} states, {len(AGE_GROUPS)} age groups')

Initialized c:\Users\User\Desktop\Data Management\cs210_national_projection\db\projection.db  |  51 states, 18 age groups


## 3. Load the CSVs into the 6 data tables

One simple helper loop handles every CSV. We use INSERT OR REPLACE so rerunning overwrites rather than failing on primary key conflicts.

In [6]:
CSV_TABLE_MAP = {
    'population_2018.csv':           ('population', False),
    'population_2023.csv':           ('population', False),
    'fertility_rates.csv':           ('fertility_rates', True),
    'mortality_rates.csv':           ('mortality_rates', True),
    'labor_force_participation.csv': ('labor_force_participation', True),
    'employment_projections.csv':    ('employment_projections', True),
    'migration_estimates.csv':       ('migration_estimates', True),
}

with sqlite3.connect(DB_PATH) as conn:
    conn.execute('PRAGMA foreign_keys = ON')
    for fname, (table, clear) in CSV_TABLE_MAP.items():
        df = pd.read_csv(DATA_DIR / fname)
        if clear:
            conn.execute(f'DELETE FROM {table}')
        cols = list(df.columns)
        sql = f"INSERT OR REPLACE INTO {table} ({','.join(cols)}) VALUES ({','.join(['?']*len(cols))})"
        records = [tuple(None if pd.isna(v) else v for v in r) for r in df.itertuples(index=False, name=None)]
        conn.executemany(sql, records)
        print(f'  {fname} -> {table}  {len(records):,} rows')
    conn.commit()

  population_2018.csv -> population  14,688 rows
  population_2023.csv -> population  14,688 rows
  fertility_rates.csv -> fertility_rates  7,344 rows
  mortality_rates.csv -> mortality_rates  14,688 rows
  labor_force_participation.csv -> labor_force_participation  14,688 rows
  employment_projections.csv -> employment_projections  408 rows
  migration_estimates.csv -> migration_estimates  14,688 rows


## 4. Sanity check

The top 5 states by 2023 population should read CA, TX, FL, NY, PA. Those are the largest US states.

In [7]:
with sqlite3.connect(DB_PATH) as conn:
    print(pd.read_sql("""
        SELECT s.state_name, SUM(p.population) AS total_2023_pop
        FROM population p JOIN states s ON s.state_code = p.state_code
        WHERE p.year = 2023 AND p.is_projection = 0
        GROUP BY p.state_code
        ORDER BY total_2023_pop DESC LIMIT 5
    """, conn).to_string(index=False))

  state_name  total_2023_pop
  California        39354846
       Texas        30808340
     Florida        22836838
    New York        19766919
Pennsylvania        13091292


The database is populated. Next, open 02_projection_pipeline.ipynb to run the Cohort Component Method.

## 5. Real-data swap (Census + BLS)

Replace the synthetic 2023 population baseline with US Census ACS B03002 data and the synthetic employment projections with BLS CES + Employment Projections values. It can be rerun safely.

In [8]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.ingest.census import build_population
from src.ingest.bls import build_employment_projections

pop_2023_real = build_population(2023)
pop_2023_real.to_csv(DATA_DIR / "population_2023.csv", index=False)
print(f"Replaced population_2023.csv with Census ACS data: {len(pop_2023_real):,} rows, {pop_2023_real.population.sum():,} total population")

emp_real = build_employment_projections()
emp_real.to_csv(DATA_DIR / "employment_projections.csv", index=False)
print(f"Replaced employment_projections.csv with BLS data: {len(emp_real):,} rows")

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("DELETE FROM population WHERE year=2023 AND is_projection=0")
    conn.execute("DELETE FROM population WHERE is_projection=1")
    conn.executemany(
        "INSERT INTO population (state_code,year,race,origin,sex,age_group,population,is_projection) VALUES (?,?,?,?,?,?,?,?)",
        list(pop_2023_real.itertuples(index=False, name=None)),
    )
    conn.execute("DELETE FROM employment_projections")
    conn.executemany(
        "INSERT INTO employment_projections (state_code,year,total_employment,source) VALUES (?,?,?,?)",
        list(emp_real.itertuples(index=False, name=None)),
    )
    conn.commit()
print("Real data loaded into DB.")

Replaced population_2023.csv with Census ACS data: 14,688 rows, 338,263,983 total population
Replaced employment_projections.csv with BLS data: 408 rows
Real data loaded into DB.
